# Evaluate: GSM8K test set

Colab front-end for `src/eval_gsm8k.py`. Greedy decoding, exact-match on the final numeric answer.

**First run the base model**, then each adapter.

## 1. Mount Drive + Clone + Install

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RUNS = '/content/drive/MyDrive/finetune-gsm8k-runs'
DRIVE_RESULTS = '/content/drive/MyDrive/finetune-gsm8k-runs/results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if not os.path.isdir('/content/finetune-gsm8k'):
    !git clone https://github.com/YuZh98/finetune-gsm8k.git /content/finetune-gsm8k
os.chdir('/content/finetune-gsm8k')
!pip install -q -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Smoke test: 16 problems against the base model

Verify the harness works before running the full 1319 problems.

In [6]:
from src.eval_gsm8k import evaluate

metrics = evaluate(adapter_path=None, limit=16, batch_size=4)
metrics

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 4/4 [03:39<00:00, 54.85s/it]


{'n_problems': 16, 'n_correct': 7, 'accuracy': 0.4375, 'elapsed_sec': 219.4}

## 3. Full eval: base model

In [7]:
from src.eval_gsm8k import evaluate, append_row
from src.config import ANSWER_REGEX

CSV_PATH = f'{DRIVE_RESULTS}/runs.csv'

metrics = evaluate(adapter_path=None, limit=None, batch_size=8)
append_row(CSV_PATH, {
    'run_id': 'run0_base',
    'adapter': '',
    'answer_regex': ANSWER_REGEX,
    **metrics,
})
print(f'✓ Base: {metrics["accuracy"]:.4f} ({metrics["n_correct"]}/{metrics["n_problems"]})')

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 165/165 [2:20:52<00:00, 51.22s/it]


✓ Base: 0.3730 (492/1319)


## 4. Full eval: trained adapters

Loops over all adapters found on Drive. Or set `RUN_IDS` manually to eval a subset.

In [9]:
from src.eval_gsm8k import evaluate, append_row
from src.config import ANSWER_REGEX

CSV_PATH = f'{DRIVE_RESULTS}/runs.csv'

# Auto-detect adapters on Drive, or set manually:
# RUN_IDS = ['run1_r8', 'run2_r16', 'run3_r64', 'run4_mlp', 'run5_lr_low', 'run6_lr_high', 'run7_data5k']
RUN_IDS = sorted([
    d for d in os.listdir(DRIVE_RUNS)
    if os.path.isdir(f'{DRIVE_RUNS}/{d}/adapter')
])
print(f'Found adapters: {RUN_IDS}')

for run_id in RUN_IDS:
    adapter = f'{DRIVE_RUNS}/{run_id}/adapter'
    print(f'\nEvaluating {run_id}...')
    metrics = evaluate(adapter_path=adapter, limit=None, batch_size=8)
    append_row(CSV_PATH, {
        'run_id': run_id,
        'adapter': adapter,
        'answer_regex': ANSWER_REGEX,
        **metrics,
    })
    print(f'  ✓ {run_id}: {metrics["accuracy"]:.4f} ({metrics["n_correct"]}/{metrics["n_problems"]})')

Found adapters: ['run1_r8', 'run2_r16', 'run3_r64', 'run4_mlp', 'run5_lr_low', 'run6_lr_high', 'run7_data5k']

Evaluating run1_r8...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 165/165 [1:58:43<00:00, 43.17s/it]


  ✓ run1_r8: 0.4162 (549/1319)

Evaluating run2_r16...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 165/165 [2:17:25<00:00, 49.97s/it]


  ✓ run2_r16: 0.3381 (446/1319)

Evaluating run3_r64...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval:  32%|███▏      | 52/165 [40:34<1:28:10, 46.81s/it]


KeyboardInterrupt: 

## 5. Inspect results

In [10]:
import pandas as pd

df = pd.read_csv(f'{DRIVE_RESULTS}/runs.csv')
df.sort_values('accuracy', ascending=False)

,run_id,adapter,answer_regex,n_problems,n_correct,accuracy,elapsed_sec
1,run1_r8,/content/drive/MyDrive/finetune-gsm8k-runs/run...,####\s*(-?\d+(?:\.\d+)?),1319,549,0.416224,7119.6
3,run1_r8,/content/drive/MyDrive/finetune-gsm8k-runs/run...,####\s*(-?\d+(?:\.\d+)?),1319,549,0.416224,7123.5
0,run0_base,NaN,####\s*(-?\d+(?:\.\d+)?),1319,492,0.373010,8372.2
2,run0_base,NaN,####\s*(-?\d+(?:\.\d+)?),1319,492,0.373010,8452.1
4,run2_r16,/content/drive/MyDrive/finetune-gsm8k-runs/run...,####\s*(-?\d+(?:\.\d+)?),1319,446,0.338135,8245.4
